In [31]:
from pathlib import Path
import os, stat

import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt

from cmdstanpy import CmdStanModel

In [32]:
## Setup
os.environ.pop('CXX', None)
os.environ.pop('CC', None)

In [33]:
out_root = './stan_output'

def compile_model(stan_path):
    print('Compiling', stan_path.name)
    mdl = CmdStanModel(stan_file=str(stan_path))
    mode = os.stat(mdl.exe_file).st_mode
    os.chmod(mdl.exe_file, mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    return mdl

def run_and_save(name, mdl, data, outdir, chains=4, warmup=1000, sampling=1000,
                 adapt_delta=0.95, max_treedepth=12, show_console=False):
    outdir.mkdir(parents=True, exist_ok=True)
    print('Running', name)

    fit = mdl.sample(
        data=data,
        chains=chains,
        parallel_chains=min(chains, os.cpu_count() or 1),
        iter_warmup=warmup,
        iter_sampling=sampling,
        seed=20260419,
        adapt_delta=adapt_delta,
        max_treedepth=max_treedepth,
        show_console=show_console,
        output_dir=str(outdir),
        save_warmup=False,
    )

    idata = az.from_cmdstanpy(
        posterior=fit,
        posterior_predictive='y_rep',
        log_likelihood='log_lik'
    )

    fn = outdir / 'idata.nc'
    idata.to_netcdf(str(fn))
    print('Saved idata to', fn)
    return fit, idata

In [34]:
## Load data
df = pd.read_csv('../data/fire_db.csv')
print(df.shape)
df.head()

(15203, 12)


,province,latitude,longitude,fire_cause,fire_type,response_type,protection_zone,fn_indicator,n_fn_20km,any_evacuation,log_fire_size_ha,log_dist_to_fn_km
0,BC,49.3014,-119.7091,human,fire,full,other,False,10,False,0.103451,-1.434898
1,BC,52.0351,-123.1770,human,fire,full,other,True,7,False,-1.335084,-2.821013
2,BC,53.2884,-120.0705,human,fire,full,other,False,0,False,-0.287845,0.671645
3,BC,52.2548,-120.8352,human,fire,full,other,False,0,False,-1.285883,-0.243862
4,BC,49.3115,-122.2438,human,fire,moderate,other,False,11,False,-1.317279,-0.808326


In [35]:
## Keep the columns needed for the spatial model
req_cols = [
    'any_evacuation',
    'log_fire_size_ha',
    'log_dist_to_fn_km',
    'fn_indicator',
    'province',
    'protection_zone',
    'latitude',
    'longitude',
]

df_sp = df.dropna(subset=req_cols).copy()

In [36]:
## Build a coarse spatial grid
LAT_STEP = 3.0
LON_STEP = 3.0

df_sp['lat_bin'] = np.floor(df_sp['latitude'] / LAT_STEP).astype(int)
df_sp['lon_bin'] = np.floor(df_sp['longitude'] / LON_STEP).astype(int)

cell_df = (
    df_sp[['lat_bin', 'lon_bin']]
    .drop_duplicates()
    .sort_values(['lat_bin', 'lon_bin'])
    .reset_index(drop=True)
    .copy()
)
cell_df['region_id'] = np.arange(1, len(cell_df) + 1)

df_sp = df_sp.merge(cell_df, on=['lat_bin', 'lon_bin'], how='left')

cell_df.head()

,lat_bin,lon_bin,region_id
0,14,-22,1
1,15,-29,2
2,15,-28,3
3,15,-27,4
4,15,-26,5


In [37]:
cell_lookup = {
    (row.lat_bin, row.lon_bin): int(row.region_id)
    for row in cell_df.itertuples(index=False)
}

edges = set()
offsets = [(-1, 0), (1, 0), (0, -1), (0, 1)]

for row in cell_df.itertuples(index=False):
    r1 = int(row.region_id)
    key = (row.lat_bin, row.lon_bin)

    for dlat, dlon in offsets:
        nbr_key = (key[0] + dlat, key[1] + dlon)
        r2 = cell_lookup.get(nbr_key)
        if r2 is not None and r1 < r2:
            edges.add((r1, r2))

edges = sorted(edges)
node1 = [e[0] for e in edges]
node2 = [e[1] for e in edges]

print('Number of regions:', len(cell_df))
print('Number of edges:', len(edges))
print('First few edges:', edges[:10])

Number of regions: 145
Number of edges: 249
First few edges: [(1, 9), (2, 3), (2, 28), (3, 4), (3, 29), (4, 5), (4, 30), (5, 6), (5, 31), (6, 7)]


In [38]:
cell_counts = df_sp.groupby('region_id').size()

print(cell_counts.describe())
cell_counts.head()

count    145.000000
mean     104.848276
std      106.878977
min        1.000000
25%       18.000000
50%       79.000000
75%      155.000000
max      513.000000
dtype: float64


region_id
1    17
2     2
3    30
4    23
5    20
dtype: int64

In [39]:
prov_cat = pd.Categorical(df_sp['province'])
prot_cat = pd.Categorical(df_sp['protection_zone'])

df_sp['prov_idx'] = prov_cat.codes + 1
df_sp['prot_idx'] = prot_cat.codes + 1

print('P =', len(prov_cat.categories))
print('K_prot =', len(prot_cat.categories))
print('R =', df_sp['region_id'].nunique())

P = 12
K_prot = 8
R = 145


In [40]:
stan_data_spatial = {
    'N': len(df_sp),
    'P': len(prov_cat.categories),
    'R': int(df_sp['region_id'].nunique()),
    'K_prot': len(prot_cat.categories),
    'E': len(edges),

    'province': df_sp['prov_idx'].astype(int).tolist(),
    'region': df_sp['region_id'].astype(int).tolist(),
    'protection_zone': df_sp['prot_idx'].astype(int).tolist(),

    'y': df_sp['any_evacuation'].astype(int).tolist(),
    'log_fire_size': df_sp['log_fire_size_ha'].astype(float).tolist(),
    'log_dist_to_fn_km': df_sp['log_dist_to_fn_km'].astype(float).tolist(),
    'fn_indicator': df_sp['fn_indicator'].astype(int).tolist(),

    'node1': node1,
    'node2': node2,
}

{k: (len(v) if isinstance(v, list) else v) for k, v in stan_data_spatial.items()}

{'N': 15203,
 'P': 12,
 'R': 145,
 'K_prot': 8,
 'E': 249,
 'province': 15203,
 'region': 15203,
 'protection_zone': 15203,
 'y': 15203,
 'log_fire_size': 15203,
 'log_dist_to_fn_km': 15203,
 'fn_indicator': 15203,
 'node1': 249,
 'node2': 249}

In [43]:
project_root = Path('../').resolve()
stan_dir = project_root / 'stan'
stan_path_spatial = stan_dir / 'spatial.stan'
mdl_spatial = compile_model(stan_path_spatial)

10:06:19 - cmdstanpy - INFO - compiling stan file /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/stan/spatial.stan to exe file /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/stan/spatial


Compiling spatial.stan


10:06:31 - cmdstanpy - INFO - compiled model executable: /Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/stan/spatial


In [ ]:
spatial_outdir = Path('./stan_output/spatial')

fit_spatial, idata_spatial = run_and_save(
    'spatial',
    mdl_spatial,
    stan_data_spatial,
    spatial_outdir,
    chains=4,
    warmup=500,
    sampling=1000,
    adapt_delta=0.95,
    show_console=False,
)

In [ ]:
az.summary(
    idata_spatial,
    var_names=[
        'alpha',
        'beta_log_fire_size',
        'beta_dist_to_fn',
        'beta_fn',
        'sigma_prov',
        'sigma_region',
    ]
)

In [ ]:
loo_spatial = az.loo(idata_spatial, pointwise=True)
print(loo_spatial)

In [48]:
# Run variational inference (ADVI) for the spatial model
spatial_vi_outdir = Path('./stan_output/spatial_vi')
spatial_vi_outdir.mkdir(parents=True, exist_ok=True)

if 'mdl_spatial' not in globals():
    if 'stan_path_spatial' not in globals():
        project_root = Path('../').resolve()
        stan_path_spatial = project_root / 'stan' / 'spatial.stan'
    mdl_spatial = compile_model(stan_path_spatial)

fit_spatial_vi = mdl_spatial.variational(
    data=stan_data_spatial,
    seed=20260419,
    output_dir=str(spatial_vi_outdir),
)

print('Saved VI outputs to', spatial_vi_outdir)
print(fit_spatial_vi)

10:33:20 - cmdstanpy - INFO - Chain [1] start processing
10:33:55 - cmdstanpy - INFO - Chain [1] done processing


Saved VI outputs to stan_output/spatial_vi
CmdStanVB: model=spatial['method=variational', 'adapt', 'engaged=1']
 csv_file:
	/Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/notebooks/stan_output/spatial_vi/spatial-20260419103320.csv
 output_file:
	/Users/pierrelardet/Documents/University_Academics/MY1/BayesianStats/Wildfires/notebooks/stan_output/spatial_vi/spatial-20260419103320_0-stdout.txt


In [49]:
# Optional: inspect generated VI output files
for p in sorted(spatial_vi_outdir.glob('*')):
    print(p)

stan_output/spatial_vi/spatial-20260419103320.csv
stan_output/spatial_vi/spatial-20260419103320_0-stdout.txt


In [ ]:
draws = fit_spatial_vi.variational_sample
if draws is None:
    draws_df = fit_spatial_vi.draws_pd()
elif isinstance(draws, pd.DataFrame):
    draws_df = draws.copy()
else:
    draws_df = pd.DataFrame(draws, columns=fit_spatial_vi.column_names)

exclude_cols = {'lp__', 'log_p__', 'log_g__'}
param_cols = [
    c for c in draws_df.columns
    if c not in exclude_cols and not c.startswith('y_rep') and not c.startswith('log_lik') and not c.startswith('u_region')
]

posterior_summary_vi = pd.DataFrame({
    'variable': param_cols,
    'mean': draws_df[param_cols].mean().values,
    'sd': draws_df[param_cols].std(ddof=1).values,
    'ci_2.5%': draws_df[param_cols].quantile(0.025).values,
    'ci_97.5%': draws_df[param_cols].quantile(0.975).values,
}).sort_values('variable').reset_index(drop=True)

posterior_summary_vi

,variable,mean,sd,ci_2.5%,ci_97.5%
0,a_prot[1],-0.506043,0.146139,-0.790179,-0.229276
1,a_prot[2],-1.495768,0.563737,-2.635433,-0.402106
2,a_prot[3],-1.953108,0.530072,-2.962978,-0.871762
3,a_prot[4],0.329030,0.307986,-0.272680,0.904382
4,a_prot[5],-1.310328,0.053563,-1.413102,-1.206196
5,a_prot[6],0.048265,0.774056,-1.444502,1.498837
6,a_prot[7],0.354493,0.678680,-0.978174,1.657410
7,a_prot[8],-0.832867,0.280857,-1.399849,-0.296771
8,a_prov[10],-1.988380,0.291886,-2.591317,-1.445350
9,a_prov[11],-0.229815,0.118186,-0.463384,0.009647


In [71]:
import re

# Build a DataFrame of VI draws
draws = fit_spatial_vi.variational_sample
if draws is None:
    draws_df_vi = fit_spatial_vi.draws_pd()
elif isinstance(draws, pd.DataFrame):
    draws_df_vi = draws.copy()
else:
    draws_df_vi = pd.DataFrame(draws, columns=fit_spatial_vi.column_names)

# Extract and order log_lik columns numerically: log_lik[1], log_lik[2], ...
ll_cols = [c for c in draws_df_vi.columns if c.startswith('log_lik[')]
ll_cols = sorted(ll_cols, key=lambda c: int(re.search(r'\[(\d+)\]', c).group(1)))

# Shape for ArviZ: (chain, draw, observation)
log_lik_arr = draws_df_vi[ll_cols].to_numpy()[None, :, :]

idata_spatial_vi = az.from_dict(
    {'log_likelihood': {'y': log_lik_arr}},
    dims={'y': ['obs_id']},
)

loo_spatial_vi = az.loo(idata_spatial_vi, var_name='y', pointwise=True, reff=1.0)
print(loo_spatial_vi)

Computed from 1000 posterior samples and 15203 observations log-likelihood matrix.

         Estimate       SE
elpd_loo -1867.83    63.33
p_loo       41.10        -
------

Pareto k diagnostic values:
                         Count   Pct.
(-Inf, 0.67]   (good)     15203  100.0%
   (0.67, 1]   (bad)          0    0.0%
    (1, Inf)   (very bad)     0    0.0%

